# MLP-VaR Softplus pruebas: ratios de volatilidad y shocks - 2010-2024, w=1
 
 Experimento para comprobar si senales relativas de cambio de regimen (`vol_5_ratio`, `vol_10_ratio`) y shocks recientes (`shock_1`, `shock_5`) reducen el clustering de violaciones sin inflar el VaR medio.
 
 Cambios frente al Softplus base:
 
 - mantiene Softplus en la salida;
 - mantiene `w=1`, ventana rolling de 900 observaciones, seed, normalizacion y funcion de perdida;
 - no usa `vol_5` ni `vol_10` en bruto;
 - anade `vol_5_ratio`, `vol_10_ratio`, `shock_1` y `shock_5` calculadas en memoria;
 - evalua todo el periodo `2010-2024` incluido;
 - guarda resumen global, detalle anual, violaciones y comparacion contra el Softplus base `w=1`.
 
 No modifica notebooks antiguos ni sobrescribe CSVs de otros experimentos.

In [ ]:
import copy
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import chi2
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

In [ ]:
DATA_DIR = Path("../data")
DATASET_PATH = DATA_DIR / "dataset_tfg.pkl"

ALPHA = 0.99
SIG = 0.05
W_LOSS = 1
WINDOW = 900
RETRAIN_EVERY = 1
SEED = 42
EVAL_START = "2010-01-01"
EVAL_END = "2024-12-31"

OUT_PRED = DATA_DIR / "nn_softplus_pruebas_shocks_2010_2024_w1.csv"
OUT_SUMMARY = DATA_DIR / "softplus_pruebas_shocks_2010_2024_w1_summary.csv"
OUT_YEARLY = DATA_DIR / "softplus_pruebas_shocks_2010_2024_w1_yearly.csv"
OUT_VIOLATIONS = DATA_DIR / "softplus_pruebas_shocks_2010_2024_w1_violations.csv"
OUT_COMPARISON = DATA_DIR / "softplus_pruebas_shocks_2010_2024_w1_comparison.csv"

## Carga de datos y nuevas features

In [ ]:
dataset = pd.read_pickle(DATASET_PATH).copy().sort_index()
assert isinstance(dataset.index, pd.DatetimeIndex), "El indice debe ser DatetimeIndex"
assert "target" in dataset.columns, "Falta target"
assert "rp_lag_0" in dataset.columns, "Falta rp_lag_0 para calcular shocks"

# Senales relativas: capturan cambio de regimen sin meter volatilidad bruta como nivel permanente.
eps = 1e-8
abs_r = dataset["rp_lag_0"].abs()
vol_5 = dataset["rp_lag_0"].rolling(5).std(ddof=0)
vol_10 = dataset["rp_lag_0"].rolling(10).std(ddof=0)
vol_20 = dataset["rp_lag_0"].rolling(20).std(ddof=0)

dataset["vol_5_ratio"] = vol_5 / (vol_20 + eps)
dataset["vol_10_ratio"] = vol_10 / (vol_20 + eps)
dataset["shock_1"] = abs_r / (vol_20 + eps)
dataset["shock_5"] = abs_r.rolling(5).max() / (vol_20 + eps)

dataset = dataset.replace([np.inf, -np.inf], np.nan).dropna().copy()

feature_cols = [c for c in dataset.columns if c != "target"]
new_features = ["vol_5_ratio", "vol_10_ratio", "shock_1", "shock_5"]
print("Dataset:", dataset.shape)
print("Rango:", dataset.index.min().date(), "->", dataset.index.max().date())
print("Features:", len(feature_cols))
print("Nuevas features incluidas:", [c for c in new_features if c in feature_cols])
print("target mean/std/min/max:", dataset["target"].mean(), dataset["target"].std(), dataset["target"].min(), dataset["target"].max())

assert len(feature_cols) == 26, f"Se esperaban 26 features tras anadir ratios y shocks, hay {len(feature_cols)}"
assert all(c in feature_cols for c in new_features), "Faltan features nuevas"
assert dataset["target"].abs().quantile(0.999) < 0.2, "El target parece tener escala inesperada"

## Utilidades de diagnostico

In [ ]:
def describe_scale(name, x):
    x = np.asarray(x, dtype=float)
    print(f"\n{name}")
    print("min:", np.nanmin(x))
    print("max:", np.nanmax(x))
    print("mean:", np.nanmean(x))
    print("p95:", np.nanpercentile(x, 95))
    print("p99:", np.nanpercentile(x, 99))
    print("p99.9:", np.nanpercentile(x, 99.9))


def plausibility_report(df, var_col="VaR_pred", loss_col="loss_real"):
    describe_scale("loss_real", df[loss_col].values)
    describe_scale(var_col, df[var_col].values)
    n_negative = int((df[var_col] < 0).sum())
    max_date = df[var_col].idxmax()
    min_date = df[var_col].idxmin()
    print("n_negative_var:", n_negative)
    print("max VaR:", max_date, float(df.loc[max_date, var_col]))
    print("min VaR:", min_date, float(df.loc[min_date, var_col]))

## Metricas de backtesting

In [ ]:
def hit_series(real_losses, var_preds):
    real_losses = np.asarray(real_losses, dtype=float).reshape(-1)
    var_preds = np.asarray(var_preds, dtype=float).reshape(-1)
    m = np.isfinite(real_losses) & np.isfinite(var_preds)
    return real_losses[m], var_preds[m], (real_losses[m] > var_preds[m]).astype(int)


def tick_loss(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    under = np.maximum(real_losses - var_preds, 0.0)
    over = np.maximum(var_preds - real_losses, 0.0)
    return float(np.mean(alpha * under + (1 - alpha) * over))


def winkler_score(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    exceedance = np.maximum(real_losses - var_preds, 0.0)
    return float(np.mean(var_preds + (2.0 / alpha) * exceedance))


def kupiec_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T, x = len(I), int(I.sum())
    p = 1 - alpha
    if T == 0 or x == 0 or x == T:
        return {"LRuc": np.nan, "p_uc": np.nan}
    p_hat = x / T
    LRuc = -2 * np.log(((1 - p) ** (T - x) * p ** x) / ((1 - p_hat) ** (T - x) * p_hat ** x))
    return {"LRuc": LRuc, "p_uc": 1 - chi2.cdf(LRuc, df=1)}


def christoffersen_cc_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    if T < 2:
        return {"LRind": np.nan, "p_ind": np.nan, "LRcc": np.nan, "p_cc": np.nan}
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        pair = (I[t - 1], I[t])
        if pair == (0, 0):
            n00 += 1
        elif pair == (0, 1):
            n01 += 1
        elif pair == (1, 0):
            n10 += 1
        else:
            n11 += 1
    if (n00 + n01) == 0 or (n10 + n11) == 0:
        LRind, p_ind = np.nan, np.nan
    else:
        pi01 = n01 / (n00 + n01)
        pi11 = n11 / (n10 + n11)
        pi = (n01 + n11) / (n00 + n01 + n10 + n11)
        L0 = ((1 - pi) ** (n00 + n10)) * (pi ** (n01 + n11))
        L1 = ((1 - pi01) ** n00) * (pi01 ** n01) * ((1 - pi11) ** n10) * (pi11 ** n11)
        if L0 <= 0 or L1 <= 0:
            LRind, p_ind = np.nan, np.nan
        else:
            LRind = -2 * np.log(L0 / L1)
            p_ind = 1 - chi2.cdf(LRind, df=1)
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    LRuc = kup["LRuc"]
    if np.isnan(LRuc) or np.isnan(LRind):
        return {"LRind": LRind, "p_ind": p_ind, "LRcc": np.nan, "p_cc": np.nan}
    LRcc = LRuc + LRind
    return {"LRind": LRind, "p_ind": p_ind, "LRcc": LRcc, "p_cc": 1 - chi2.cdf(LRcc, df=2)}


def dq_test(real_losses, var_preds, alpha=0.99, lags=4):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    p = 1 - alpha
    if T <= lags + 5:
        return {"DQ": np.nan, "p_dq": np.nan}
    hit = I - p
    y = hit[lags:]
    X = np.column_stack([np.ones(len(y))] + [hit[lags - j:T - j] for j in range(1, lags + 1)])
    XtX = X.T @ X
    if np.linalg.matrix_rank(XtX) < XtX.shape[0]:
        return {"DQ": np.nan, "p_dq": np.nan}
    beta = np.linalg.inv(XtX) @ X.T @ y
    resid = y - X @ beta
    sigma2 = (resid @ resid) / len(y)
    if sigma2 <= 0:
        return {"DQ": np.nan, "p_dq": np.nan}
    DQ = float((beta.T @ XtX @ beta) / sigma2)
    return {"DQ": DQ, "p_dq": 1 - chi2.cdf(DQ, df=X.shape[1])}

In [ ]:
def evaluate_var_model(df, alpha=0.99, sig=0.05):
    real_losses = df["loss_real"].values
    var_preds = df["VaR_pred"].values
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    violations = int(I.sum())
    violation_rate = violations / T
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
    dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
    df_2020 = df.loc["2020"] if (df.index.year == 2020).any() else pd.DataFrame(columns=df.columns)
    row = {
        "Model": "Softplus + ratios + shocks",
        "w": W_LOSS,
        "T": T,
        "Expected Viol.": (1 - alpha) * T,
        "Violations": violations,
        "Violation Rate": violation_rate,
        "VR Gap": abs(violation_rate - (1 - alpha)),
        "Coverage Bias": violation_rate - (1 - alpha),
        "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
        "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
        "Mean VaR Level": float(np.nanmean(var_preds)),
        "Max VaR Level": float(np.nanmax(var_preds)),
        "Min VaR Level": float(np.nanmin(var_preds)),
        "Mean VaR 2020": float(df_2020["VaR_pred"].mean()) if len(df_2020) else np.nan,
        "Max VaR 2020": float(df_2020["VaR_pred"].max()) if len(df_2020) else np.nan,
        "n_negative_var": int((df["VaR_pred"] < 0).sum()),
        "p_UC": kup["p_uc"],
        "UC Test": "PASS" if kup["p_uc"] > sig else "FAIL",
        "p_CC": cc["p_cc"],
        "CC Test": "PASS" if cc["p_cc"] > sig else "FAIL",
        "p_DQ": dq["p_dq"],
        "DQ Test": "PASS" if dq["p_dq"] > sig else "FAIL",
    }
    vals = [row["p_UC"], row["p_CC"], row["p_DQ"]]
    row["p_Mean"] = float(np.mean([v for v in vals if pd.notnull(v)]))
    row["Valid(CC&DQ)"] = "YES" if row["CC Test"] == "PASS" and row["DQ Test"] == "PASS" else "NO"
    return pd.DataFrame([row])


def evaluate_by_year(df, alpha=0.99):
    rows = []
    for year, g in df.groupby("year"):
        real_losses = g["loss_real"].values
        var_preds = g["VaR_pred"].values
        violations = int(g["viol"].sum())
        expected = (1 - alpha) * len(g)
        kup = kupiec_test(real_losses, var_preds, alpha=alpha)
        cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
        dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
        rows.append({
            "year": int(year),
            "T": len(g),
            "Expected Viol.": expected,
            "Violations": violations,
            "Violation Rate": violations / len(g),
            "VR Gap": abs((violations / len(g)) - (1 - alpha)),
            "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
            "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
            "Mean VaR Level": float(np.nanmean(var_preds)),
            "Max VaR Level": float(np.nanmax(var_preds)),
            "Min VaR Level": float(np.nanmin(var_preds)),
            "Max Loss": float(np.nanmax(real_losses)),
            "Mean Loss": float(np.nanmean(real_losses)),
            "n_negative_var": int((g["VaR_pred"] < 0).sum()),
            "p_UC": kup["p_uc"],
            "UC Test": "PASS" if kup["p_uc"] > SIG else "FAIL",
            "p_CC": cc["p_cc"],
            "CC Test": "PASS" if cc["p_cc"] > SIG else "FAIL",
            "p_DQ": dq["p_dq"],
            "DQ Test": "PASS" if dq["p_dq"] > SIG else "FAIL",
        })
    return pd.DataFrame(rows)


def violation_table(df):
    cols = ["VaR_pred", "loss_real", "viol", "year"]
    out = df.loc[df["viol"] == 1, cols].copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["exceedance_ratio"] = out["loss_real"] / out["VaR_pred"]
    return out.sort_index()


def worst_days_table(df, n=25):
    out = df.copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["loss_minus_var"] = out["exceedance"]
    return out.sort_values("loss_real", ascending=False).head(n)

## Modelo Softplus

In [ ]:
def inverse_softplus(y):
    return math.log(math.expm1(y))


def var_loss(y_true, y_pred, q=0.99, w=1.0):
    e = y_true - y_pred
    weight = torch.where(e > 0, torch.as_tensor(w, dtype=y_pred.dtype, device=y_pred.device), torch.ones_like(e))
    pinball = torch.maximum(q * e, (q - 1) * e)
    return torch.mean(weight * pinball)


class MLPVaRSoftplus(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.softplus = nn.Softplus()
        nn.init.constant_(self.body[-1].bias, inverse_softplus(0.02))

    def forward(self, x):
        return self.softplus(self.body(x))


def train_one_model(X_train, y_train, d_in, seed, w_loss, alpha=0.99, max_epochs=200, lr=5e-5, batch_size=256, patience=15, val_split=0.10):
    torch.manual_seed(seed)
    np.random.seed(seed)
    n = len(X_train)
    split = int(n * (1 - val_split))
    X_tr, X_val = X_train[:split], X_train[split:]
    y_tr, y_val = y_train[:split], y_train[split:]
    model = MLPVaRSoftplus(d_in=d_in)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=batch_size, shuffle=False)
    X_val_t = torch.from_numpy(X_val)
    y_val_t = torch.from_numpy(y_val)
    best_loss = np.inf
    best_state = None
    patience_counter = 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            pred = model(xb)
            loss_val = var_loss(yb, pred, q=alpha, w=w_loss)
            loss_val.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = var_loss(y_val_t, val_pred, q=alpha, w=w_loss).item()
        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

## Ejecucion rolling 2010-2024

In [ ]:
eval_dates = dataset.loc[pd.Timestamp(EVAL_START):pd.Timestamp(EVAL_END)].index
var_preds = []
real_targets = []
dates = []
current_model = None
muX, sdX = None, None

for i, current_date in enumerate(tqdm(eval_dates, desc="Softplus ratios/shocks w=1", dynamic_ncols=True)):
    if i % RETRAIN_EVERY == 0:
        train_end = current_date - pd.Timedelta(days=1)
        train_df = dataset.loc[:train_end].tail(WINDOW)
        if len(train_df) < 250:
            if current_model is None:
                continue
        else:
            X_train = train_df[feature_cols].values.astype(np.float32)
            y_train = train_df["target"].values.astype(np.float32).reshape(-1, 1)
            muX = X_train.mean(axis=0, keepdims=True)
            sdX = X_train.std(axis=0, keepdims=True) + 1e-8
            X_train = (X_train - muX) / sdX
            current_model = train_one_model(X_train, y_train, d_in=X_train.shape[1], seed=SEED, w_loss=W_LOSS, alpha=ALPHA)

    if current_model is None or muX is None or sdX is None:
        continue

    test_df = dataset.loc[[current_date]]
    X_test = (test_df[feature_cols].values.astype(np.float32) - muX) / sdX
    y_test = test_df["target"].values.astype(np.float32).reshape(-1, 1)

    current_model.eval()
    with torch.no_grad():
        pred = current_model(torch.from_numpy(X_test)).numpy().reshape(-1)[0]

    var_preds.append(float(pred))
    real_targets.append(float(y_test.reshape(-1)[0]))
    dates.append(current_date)

pred_df = pd.DataFrame({"VaR_pred": var_preds, "loss_real": real_targets}, index=pd.DatetimeIndex(dates)).sort_index()
pred_df = pred_df.loc[EVAL_START:EVAL_END].dropna().copy()
pred_df["viol"] = (pred_df["loss_real"] > pred_df["VaR_pred"]).astype(int)
pred_df["year"] = pred_df.index.year

pred_df.to_csv(OUT_PRED)
print("Guardado:", OUT_PRED)
display(pred_df.head())
display(pred_df.tail())

## Resultados y comparacion con Softplus base w=1

In [ ]:
plausibility_report(pred_df)

summary = evaluate_var_model(pred_df, alpha=ALPHA, sig=SIG)
yearly = evaluate_by_year(pred_df, alpha=ALPHA)
violations = violation_table(pred_df)

summary.to_csv(OUT_SUMMARY, index=False)
yearly.to_csv(OUT_YEARLY, index=False)
violations.to_csv(OUT_VIOLATIONS)

print("Guardado:", OUT_SUMMARY)
print("Guardado:", OUT_YEARLY)
print("Guardado:", OUT_VIOLATIONS)

print("\nResumen global 2010-2024")
display(summary)

print("\nDiagnostico anual completo")
display(yearly)

print("\nAnos ordenados por tasa de violacion")
display(yearly.sort_values(["Violation Rate", "Violations"], ascending=False))

print("\nDetalle COVID 2020")
display(pred_df.loc["2020"].describe())
display(yearly.loc[yearly["year"] == 2020])

print("\nViolaciones completas")
display(violations)

print("\nPeores dias por perdida realizada")
display(worst_days_table(pred_df, n=25))

In [ ]:
# Comparacion directa contra el Softplus base w=1 combinando validacion 2010-2014 y test 2015-2024.
base_paths = [DATA_DIR / "nn_softplus_validation_w1.csv", DATA_DIR / "nn_softplus_test_w1.csv"]
base_parts = []
for path in base_paths:
    if path.exists():
        part = pd.read_csv(path, index_col=0, parse_dates=True)
        base_parts.append(part)
        print("Base cargada:", path, part.index.min().date(), "->", part.index.max().date(), part.shape)

if base_parts:
    base_df = pd.concat(base_parts).sort_index()
    base_df = base_df.loc[EVAL_START:EVAL_END]
    base_df = base_df[~base_df.index.duplicated(keep="last")].copy()
    base_df["viol"] = (base_df["loss_real"] > base_df["VaR_pred"]).astype(int)
    base_df["year"] = base_df.index.year

    base_summary = evaluate_var_model(base_df, alpha=ALPHA, sig=SIG)
    base_summary["Model"] = "Softplus base w=1"
    cmp = pd.concat([base_summary, summary], ignore_index=True)
    cmp.to_csv(OUT_COMPARISON, index=False)
    print("Guardado:", OUT_COMPARISON)

    print("\nComparacion global")
    display(cmp[[
        "Model", "w", "T", "Violations", "Violation Rate", "VR Gap", "Coverage Bias",
        "Tick Loss", "Winkler Score", "Mean VaR Level", "Max VaR Level", "Min VaR Level",
        "Mean VaR 2020", "Max VaR 2020", "n_negative_var", "p_UC", "UC Test", "p_CC", "CC Test", "p_DQ", "DQ Test", "p_Mean", "Valid(CC&DQ)",
    ]])

    base_yearly = evaluate_by_year(base_df, alpha=ALPHA).add_prefix("base_")
    new_yearly = yearly.add_prefix("new_")
    yearly_cmp = base_yearly.merge(new_yearly, left_on="base_year", right_on="new_year", how="outer")
    yearly_cmp["delta_violations_new_minus_base"] = yearly_cmp["new_Violations"] - yearly_cmp["base_Violations"]
    yearly_cmp["delta_rate_new_minus_base"] = yearly_cmp["new_Violation Rate"] - yearly_cmp["base_Violation Rate"]
    yearly_cmp["delta_tick_new_minus_base"] = yearly_cmp["new_Tick Loss"] - yearly_cmp["base_Tick Loss"]
    yearly_cmp["delta_winkler_new_minus_base"] = yearly_cmp["new_Winkler Score"] - yearly_cmp["base_Winkler Score"]

    print("\nComparacion anual base vs nuevo")
    display(yearly_cmp)

    print("\nViolaciones base")
    display(violation_table(base_df))
else:
    print("No existen CSVs base w=1; se omite comparacion contra Softplus base.")

## Lectura esperada
 
 Comparar la fila `Softplus + ratios + shocks` contra `Softplus base w=1` y revisar tambien la tabla anual:
 
 - Objetivo principal: que `UC`, `CC` y `DQ` no fallen globalmente, con especial atencion a `DQ`.
 - La cobertura debe quedar cerca de 1%, no muy por debajo como en el experimento con `vol_5`/`vol_10` en bruto.
 - `Mean VaR Level` y `Max VaR Level` no deberian inflarse claramente frente al Softplus base.
 - En 2020 buscamos menos clustering sin crear un VaR maximo economicamente absurdo.
 - Si `n_negative_var` sigue en cero, Softplus mantiene la restriccion positiva.